# ML Assignment 2: Email Spam Classification

Compare five models using the [UCI Spambase dataset](https://archive.ics.uci.edu/dataset/94/spambase).
Run the cells in order from the Assignment_2 folder.

In [2]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

## 1. Load the data

In [4]:
folder = Path.cwd()
names = (folder / "data" / "spambase.names").read_text().splitlines()
features = [line.split(":")[0] for line in names if "continuous." in line]
data = pd.read_csv(folder / "data" / "spambase.data", header=None,
                   names=features + ["spam"])

print("Rows:", len(data))
print("Input features:", len(features))
print("Missing values:", data.isna().sum().sum())
print("Repeated rows:", data.duplicated().sum())
assert len(features) >= 12 and len(data) >= 500
assert np.isfinite(data.to_numpy()).all()
assert set(data["spam"].unique()) == {0, 1}

Rows: 4601
Input features: 57
Missing values: 0
Repeated rows: 391


## 2. Prepare the data

Remove repeated rows, then remove rows with the same inputs but different labels.

In [6]:
data = data.drop_duplicates()
conflicting = data.duplicated(subset=features, keep=False)
print("Rows with conflicting labels:", conflicting.sum())
data = data.loc[~conflicting].reset_index(drop=True)
print("Rows used:", len(data))
data["spam"].value_counts().sort_index()

Rows with conflicting labels: 6
Rows used: 4204


spam
0    2528
1    1676
Name: count, dtype: int64

## 3. Split the data

Use 80% for training and 20% for testing. Keep similar class proportions in both parts.

In [8]:
X = data[features]
y = data["spam"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Training rows: 3363
Test rows: 841


## 4. Train the models

Scaling is fitted only on training data and saved with Logistic Regression, KNN and SVM.

In [10]:
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": make_pipeline(StandardScaler(), SVC())
}
model_files = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Support Vector Machine": "svm.joblib"
}
(folder / "model").mkdir(exist_ok=True)
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    if name == "Support Vector Machine":
        scores = model.decision_function(X_test)
    else:
        scores = model.predict_proba(X_test)[:, 1]

    results.append({
        "ML Model Name": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "AUC": roc_auc_score(y_test, scores),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "MCC": matthews_corrcoef(y_test, predictions)
    })
    joblib.dump(model, folder / "model" / model_files[name])

## 5. Compare the results

Spam (1) is the positive class. AUC uses probabilities, or the SVM decision scores.

In [12]:
comparison = pd.DataFrame(results).set_index("ML Model Name")
comparison.round(4)

,Accuracy,AUC,Precision,Recall,F1,MCC
ML Model Name,,,,,,
Logistic Regression,0.9239,0.9721,0.9045,0.9045,0.9045,0.8412
Decision Tree,0.9227,0.9186,0.9066,0.8985,0.9025,0.8385
KNN,0.9168,0.9654,0.9027,0.8866,0.8946,0.8259
Naive Bayes,0.8252,0.9498,0.7035,0.9701,0.8156,0.6881
Support Vector Machine,0.9310,0.9765,0.9184,0.9075,0.9129,0.8559


## 6. Save the test data

Keep the input columns and the correct labels together for the app.

In [14]:
test_data = pd.concat([X_test, y_test], axis=1)
test_data.to_csv(folder / "test_data.csv", index=False)
print("Saved test_data.csv and all five models in model/.")

Saved test_data.csv and all five models in model/.
